# Pydantic AI with Ollama and MCP Server

This notebook demonstrates how to use Pydantic AI with Ollama models and MCP servers for DNA sequence analysis.

# Install dependencies
```bash
uv pip install pydantic-ai
uv pip install nest-asyncio
```

# Ollama Model is available
```bash
ollama models list
```
Please make sure qwen3.6:latest is available. if not, please run `ollama pull qwen3.6:latest`. Or choose another model and modify the model name in the code.

In [1]:
import nest_asyncio
import asyncio
import time
nest_asyncio.apply()

In [2]:
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP





ollama_model = OpenAIChatModel(
    model_name='qwen3.6:latest',
    provider=OllamaProvider(base_url='http://localhost:11434/v1'),
)


In [3]:
# Create MCP server connection
server = MCPServerStreamableHTTP('http://localhost:8000/mcp')

In [4]:
# Create agent with MCP server tools and proper system prompt
agent_ollama = Agent(
    ollama_model, 
    toolsets=[server],
    system_prompt='''You are a DNA analysis assistant with access to specialized DNA analysis tools via MCP server.

When analyzing a DNA sequence, you should:
1. First call _list_loaded_models to see what models are available
2. Then call dna_multi_model_predict with the DNA sequence and appropriate model names
3. Interpret and explain the results in a comprehensive way, not just only list the results

Available tools should include:
- _list_loaded_models: Lists available DNA analysis models
- dna_multi_model_predict: Predicts DNA sequence properties using multiple models

Always use the tools to provide accurate analysis. Based on the returned results, make reasonable inferences with comprehensive biological functions of this sequence.'''
)

In [5]:
# Analyze DNA sequence using MCP server with proper async context
async def analyze_dna_sequence():
    async with agent_ollama:  # This ensures proper MCP server connection
        result = await agent_ollama.run(
            'What is the function of following DNA sequence? Please analyze it thoroughly using all available models: AGAAAAAACATGACAAGAAATCGATAATAATACAAAAGCTATGATGGTGTGCAATGTCCGTGTGCATGCGTGCACGCATTGCAACCGGCCCAAATCAAGGCCCATCGATCAGTGAATACTCATGGGCCGGCGGCCCACCACCGCTTCATCTCCTCCTCCGACGACGGGAGCACCCCCGCCGCATCGCCACCGACGAGGAGGAGGCCATTGCCGGCGGCGCCCCCGGTGAGCCGCTGCACCACGTCCCTGA'
        )
        return result

# Run the analysis
result = await analyze_dna_sequence()

time.sleep(3)

In [6]:
# Display the comprehensive analysis result
print("=== DNA Sequence Analysis Result ===")
print(result.output)
print("\n=== Usage Statistics ===")
print(result.usage())


=== DNA Sequence Analysis Result ===
Excellent! I've analyzed your DNA sequence using all three available models. Let me provide a comprehensive analysis:

## 🧬 **Multi-Model Analysis Results**

### **1. Promoter Analysis** (promoter_model - DNABERT-based)
- **Prediction**: Core promoter detected with **94.68% confidence**
- **Alternative**: Not promoter (5.32%)
- **Interpretation**: This sequence has strong promoter characteristics, suggesting it plays a key role in transcription initiation.

### **2. Evolutionary Conservation** (conservation_model - DNABERT-based)
- **Prediction**: Conserved region with **83.76% confidence**
- **Alternative**: Not conserved (16.24%)
- **Interpretation**: This sequence has been preserved through evolution, strongly indicating functional importance.

### **3. Chromatin Accessibility** (open_chromatin_model - DNAMamba-based)
- **Prediction**: Mixed between **Full open (49.90%)** and **Partial open (49.71%)**
- **Not open**: Only 0.39%
- **Interpretation

In [7]:
# Alternative approach: Test individual tool calls to debug
async def test_individual_tools():
    async with agent_ollama:
        # First, let's try to list available tools
        print("=== Testing tool availability ===")
        
        # Try to get the agent to use the list models tool
        list_result = await agent_ollama.run(
            'Please list all the available DNA analysis models using the _list_loaded_models tool.'
        )
        print("List models result:")
        print(list_result.output)
        
        return list_result

# Run individual tool test
list_result = await test_individual_tools()

=== Testing tool availability ===


List models result:
Here are all the available DNA analysis models loaded in the system:

## **Available DNA Analysis Models**

### 1. **promoter_model** 🎯
- **Purpose**: Promoter prediction (identifies core promoter regions in DNA sequences)
- **Architecture**: DNABERT with BPE tokenizer
- **Task Type**: Binary classification
- **Classes**: "Not promoter" vs "Core promoter"
- **Species Focus**: Plant
- **Performance**:
  - Accuracy: 85%
  - F1 Score: 82%
  - Precision: 80%
  - Recall: 85%
- **Memory Usage**: ~369.3 MB

### 2. **conservation_model** 🧬
- **Purpose**: Evolutionary conservation prediction (identifies functionally important sequences)
- **Architecture**: DNABERT with BPE tokenizer
- **Task Type**: Binary classification
- **Classes**: "Not conserved" vs "Conserved"
- **Species Focus**: Plant
- **Performance**:
  - Accuracy: 88%
  - F1 Score: 85%
  - Precision: 83%
  - Recall: 87%
- **Memory Usage**: ~369.3 MB

### 3. **open_chromatin_model** 🌿
- **Purpose**: Chromatin acces